In [ ]:
# Problem Statement:
# Imagine Walmart wants to build a Retail Assistant AI that can help store managers make operational decisions.
# For example, a manager at a Bengaluru store asks:
#     “Based on today’s weather and current market demand, which products should I stock more of today?”
# The AI cannot answer reliably using only its existing LLM knowledge because the answer depends on live information such as today's weather and current demand trends.

# So the AI needs to connect to external systems such as:
# a. OpenWeatherMap → current weather
# b. Tavily Search → current market/demand signals
# c. OpenAI LLM → reasoning and final recommendation

# The notebook then asks three important architect-level questions.
# 1. How should the AI connect to external tools — REST API or MCP?
#     At Walmart scale, should we continue connecting agents directly through REST, or standardize tool access through MCP?
# 2. Which framework should orchestrate the AI agent?
#     The notebook implements essentially the same Walmart use case in three different ways: Python-only, LangChain, and LangGraph.
#     Should Walmart build the orchestration using plain Python, LangChain, or LangGraph?
# 3. Should Walmart build each AI component itself or buy/use an existing solution?

In [ ]:
# Load the libraries and keys needed for the full notebook.
# This cell also defines the store we will use in every example.
import os
import json
import time
import requests
from typing import TypedDict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

OPENAI_KEY  = os.getenv('OPENAI_API_KEY')
WEATHER_KEY = os.getenv('OPENWEATHERMAP_API_KEY')
TAVILY_KEY  = os.getenv('TAVILY_API_KEY')

client = OpenAI(api_key=OPENAI_KEY)

# Check whether all required API keys are present before we continue.
missing = [k for k, v in {
    'OPENAI_API_KEY':         OPENAI_KEY,
    'OPENWEATHERMAP_API_KEY': WEATHER_KEY,
    'TAVILY_API_KEY':         TAVILY_KEY,
}.items() if not v]

if missing:
    print(f'WARNING: Missing API keys: {missing}')
    print('Add them to your .env file before running this notebook.')
else:
    print('All required API keys loaded.')

# Fixed store context keeps every comparison in the notebook consistent.
STORE_ID   = 'WMT-2847'
STORE_CITY = 'Bengaluru'
print(f'Store context: {STORE_ID} | {STORE_CITY}, India')

## The Decision Landscape

Every production AI system at Walmart scale requires three interlocking decisions made before any code is written:

1. **Protocol selection:** How should your AI agent communicate with external systems? REST API or Model Context Protocol (MCP)?
2. **Framework selection:** What orchestration layer do you build on? LangChain, LangGraph, or Python-only?
3. **Build vs Buy:** For each component, is it cheaper to build it or to buy a vendor solution?

Each decision compounds. A wrong protocol choice forces a framework rewrite downstream.

**The running scenario throughout this notebook:**

> *You are the AI engineer for the Walmart India Retail Assistant deployed at 4,700 stores with 50,000+ queries per day. The store manager at WMT-2847 in Bengaluru asks: "Based on today's actual conditions, what should we prioritise stocking today?"*

Every tool call in this notebook returns **live data** from real external APIs. The recommendation changes based on real weather and real market signals retrieved at runtime.

## Core API Functions

These two functions are the live data foundation used across all three sections.
Both make real HTTP calls to external services every time they are invoked.

In [ ]:
def fetch_weather(city: str, country_code: str = 'IN') -> dict:
    # Call the live OpenWeatherMap API for current weather.
    url    = 'https://api.openweathermap.org/data/2.5/weather'
    params = {'q': f'{city},{country_code}', 'appid': WEATHER_KEY, 'units': 'metric'}
    resp   = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    d = resp.json()
    print(f"Current weather is {d}")
    return {
        'city':           d['name'],
        'country':        d['sys']['country'],
        'temperature_c':  round(d['main']['temp'], 1),
        'feels_like_c':   round(d['main']['feels_like'], 1),
        'humidity_pct':   d['main']['humidity'],
        'condition':      d['weather'][0]['description'],
        'condition_main': d['weather'][0]['main'],
        'wind_speed_ms':  d['wind']['speed'],
        'pressure_hpa':   d['main']['pressure'],
    }

fetch_weather("Mumbai")

In [ ]:
def fetch_demand_trends(query: str, max_results: int = 3) -> dict:
    # Call the live Tavily API for current market demand signals.
    resp = requests.post(
        'https://api.tavily.com/search',
        json={
            'api_key':        TAVILY_KEY,
            'query':          query,
            'max_results':    max_results,
            'search_depth':   'basic',
            'include_answer': True,
        },
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    return {
        'answer':  data.get('answer', ''),
        'results': [
            {'title': r['title'], 'content': r['content'][:350]}
            for r in data.get('results', [])
        ],
    }
fetch_demand_trends("What are the latest updates on US-Israel vs Iran war as of 23rd September 2026?")

## Section 1: REST API Integration -- Live External Services

In the REST pattern the developer manually writes a natural-language tool description and the AI uses that description to decide when and how to call the tool.

**Strengths:** Mature ecosystem, no new dependencies, works with any HTTP service.
**Weakness:** Tool schema lives in the developer's prompt. Schema drift and multi-agent reuse require copy-pasting descriptions across every agent that needs the tool.

The two tools below call **real external APIs** on every invocation:
- `get_store_weather` -- live call to OpenWeatherMap
- `search_demand_trends` -- live call to Tavily Search

In [ ]:
# This cell shows the REST approach.
# The developer writes the tool schema by hand and the model uses it.
# REST tool schema -- developer writes this by hand for each agent that needs it
REST_TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_store_weather',
            'description': (
                'Get real-time weather at a Walmart India store location. '
                'Use this to identify weather-driven demand: '
                'rain drives umbrella/raincoat/waterproof footwear sales, '
                'heat drives cold beverages/ice cream/sunscreen sales, '
                'cold drives hot beverages/heaters/blanket sales.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'city':         {'type': 'string', 'description': 'City where the Walmart store is located'},
                    'country_code': {'type': 'string', 'description': 'ISO country code (default IN for India)'},
                },
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'search_demand_trends',
            'description': (
                'Search for real-time retail product demand trends and market signals using Tavily. '
                'Use this to identify high-demand product categories based on current market conditions.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'query':       {'type': 'string', 'description': 'Search query for demand or market signals'},
                    'max_results': {'type': 'integer', 'description': 'Number of results to return (1-5)'},
                },
                'required': ['query'],
            },
        },
    },
]


def execute_rest_tool(name: str, args: dict) -> str:
    # Route each REST tool call to the matching Python function.
    if name == 'get_store_weather':
        result = fetch_weather(args['city'], args.get('country_code', 'IN'))
    elif name == 'search_demand_trends':
        result = fetch_demand_trends(args['query'], args.get('max_results', 3))
    else:
        result = {'error': f'Unknown tool: {name}'}
    return json.dumps(result)


def walmart_rest_agent(query: str) -> dict:
    # Run a full REST-style agent loop with live tool calls.
    start    = time.time()
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                'Use available tools to retrieve live data before making recommendations. '
                'Base your answer entirely on the real data returned by the tools.'
            ),
        },
        {'role': 'user', 'content': query},
    ]

    tools_called = []
    total_in = total_out = 0

    # Keep calling the model until it stops asking for tools.
    for _ in range(6):
        resp = client.chat.completions.create(
            model='gpt-4o-mini', messages=messages, tools=REST_TOOLS,
            tool_choice='auto', temperature=0, max_tokens=500,
        )
        total_in  += resp.usage.prompt_tokens
        total_out += resp.usage.completion_tokens
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final_answer = msg.content.strip()
            break

        messages.append(msg)
        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_rest_tool(tc.function.name, args)
            tools_called.append({'tool': tc.function.name, 'args': args})
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

    latency = time.time() - start
    cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
    return {
        'answer':       final_answer,
        'tools_called': tools_called,
        'latency_sec':  round(latency, 2),
        'cost_usd':     round(cost, 6),
        'tokens_in':    total_in,
        'tokens_out':   total_out,
        'protocol':     'REST',
    }

# Store manager asks question
#         ↓
# GPT receives question
#         ↓
# GPT sees available REST tool definitions
#         ↓
# GPT asks for Bengaluru weather
#         ↓
# Python calls Weather API
#         ↓
# Weather result returned to GPT
#         ↓
# GPT asks for current demand trends
#         ↓
# Python calls Tavily
#         ↓
# Market results returned to GPT
#         ↓
# GPT combines weather + market signals
#         ↓
# GPT generates stocking recommendation
#         ↓
# System records latency, token usage and cost

In [ ]:
STORE_ID, STORE_CITY

In [ ]:
# Ask the REST agent the main Walmart store question and inspect the result.
STORE_QUERY = (
    f'I am the store manager at {STORE_ID} in {STORE_CITY}. '
    "Based on today's actual weather conditions and current market demand signals, "
    'give me 3 specific product categories I should prioritise stocking today. '
    'For each category, explain why the live data supports this recommendation.'
)

print(f'Query: {STORE_QUERY}')
print()

rest_result = walmart_rest_agent(STORE_QUERY)

# Show which tools were used before we read the final answer.
print('Tools called by the REST agent:')
for t in rest_result['tools_called']:
    print(f'  {t["tool"]}({json.dumps(t["args"])})')

print()
print('REST Agent Answer (based on live data):')
print('-' * 70)
print(rest_result['answer'])
print('-' * 70)
print(f'Latency : {rest_result["latency_sec"]}s')
print(f'Cost    : ${rest_result["cost_usd"]}')
print(f'Protocol: REST (hand-written tool schema)')

In [ ]:
# REST works perfectly well for connecting an agent to APIs, but at enterprise scale the architectural question becomes how to centralize and govern those tools rather than allowing every agent team to independently duplicate tool definitions and integration logic.

## Section 2: Model Context Protocol (MCP) -- Real FastMCP Server

MCP (Anthropic, 2024) separates tool definition from tool calling.
The MCP server owns the schema. Any MCP-compatible AI client connects, discovers tools automatically, and calls them -- without the developer writing a single word of tool description.

**Architecture:**
```
AI Host (GPT-4o-mini)  <-->  MCP Client (test_client)  <-->  FastMCP Server  <-->  Live External APIs
```

**What changes vs REST:**
- Tool descriptions live on the **server**, not in developer prompts
- Schema is **machine-readable** JSON -- no natural language required
- Any number of AI clients can point at the same server without duplicating schema
- The server handles versioning; clients auto-discover changes on reconnect

**What stays the same:**
- Underlying HTTP calls to OpenWeatherMap and Tavily are identical
- OpenAI token cost per call is the same
- MCP adds a schema-fetch round-trip on first connection (~5ms)

The FastMCP server below uses real API calls inside every registered tool.

In [ ]:
# Why the REST approach becomes difficult for AI agents

# REST itself is not bad. REST is still one of the most widely used ways for systems to communicate.

# The problem is that when you use REST with an AI agent, the AI application usually has to manually understand and describe every API it can use.

# REST_TOOLS = [
#     get_store_weather,
#     search_demand_trends
# ]
# and for every tool we manually defined:
#     Tool name
#     Tool description
#     Input parameters
#     Required parameters
#     Routing logic

# Then we separately wrote:
#     execute_rest_tool()

# For two tools, this is easy.
# But imagine Walmart has 100 AI agents and 200 enterprise capabilities:
#     Inventory
#     Pricing
#     Weather
#     Orders
#     Suppliers
#     Promotions
#     Returns
#     Store traffic
#     Employee scheduling
#     Product catalog
#     Shipping
#     Warehouse
#     Customer reviews

# Every AI application may start writing its own integration code.
# You could end up with:
# Agent 1 → custom weather integration
# Agent 2 → another weather integration
# Agent 3 → another weather integration
# Agent 4 → another weather integration

# The real problem becomes:
#     Too many custom AI-to-system integrations.

# Anthropic described exactly this issue when it introduced MCP: every new data source often required its own custom implementation, which made connected AI systems difficult to scale.

In [ ]:
# MCP stands for Model Context Protocol.

# MCP is a standard way for AI applications to discover and use external tools and data.

# Instead of every AI application inventing its own way of describing:
#     "Here is my weather tool."
#     "Here are its parameters."
#     "This is how you call it."
# an MCP server can expose that information using a common standard.

# Before USB became common, different devices needed different connections.

# You might have had:
# Keyboard → one connector
# Mouse → another connector
# Printer → another connector
# Camera → another connector

# USB created a common connection standard.
# MCP is trying to solve a similar problem for AI systems.
# Before MCP:
# Claude → custom Salesforce integration
# ChatGPT app → another Salesforce integration
# Internal Walmart agent → another Salesforce integration
# IDE assistant → another Salesforce integration

# Lots of:
#     AI application × enterprise system combinations.
# That creates what architects sometimes call an N × M integration problem.

# 10 AI applications
# ×
# 20 enterprise systems
# = potentially many custom integrations

# Enterprise capability
#         ↓
#      MCP Server
#         ↓
# standard MCP interface
#         ↓
# different MCP-compatible AI applications

In [ ]:
# With MCP, Walmart could create: Walmart Inventory MCP Server
# which exposes:
#     check_inventory
#     get_low_stock_products
#     get_store_inventory

# Then:
# Store Manager AI ─────┐
# Supply Chain AI ──────┼──→ Inventory MCP Server
# Merchandising AI ─────┘

# All three can discover the same governed tool definitions.

In [ ]:
# Is MCP better than REST?

# REST and MCP solve different problems.

# REST primarily answers:
#     How can one software system communicate with another software system over HTTP?

# MCP answers more specifically:
#     How can an AI application discover and interact with tools, resources and external systems using a standard AI-oriented protocol?

# So REST might expose:
#     GET /stores/2847/weather

# while MCP exposes an AI-facing tool such as:
# Tool:
# get_store_weather

# Description:
# Get current weather for a Walmart store.

# Input:
# city: string
# country_code: string

# The MCP layer is giving the AI the meaning and contract it needs.

# LLM / AI Agent
#       ↓
# MCP Client
#       ↓
# MCP Server
#       ↓
# Existing REST API
#       ↓
# Walmart Inventory System

In [ ]:
# Agent:
# "Check inventory for umbrellas."

#         ↓

# MCP:
# check_inventory(product="umbrella")

#         ↓

# MCP server internally calls:

# GET /api/inventory?product=umbrella

#         ↓

# Existing REST service

In [ ]:
# %pip install mcp -q

In [ ]:
# This cell sets up the FastMCP server and registers its tools.
# The main idea is that tool definitions live on the server side.
MCP_AVAILABLE = False

try:
    import inspect # inspect is a Python utility that allows the program to examine Python functions dynamically.
    # from mcp.server.fastmcp import FastMCP
    from mcp.server.mcpserver import MCPServer

    # This helps async-friendly packages behave inside Jupyter.
    # Jupyter async compatibility
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass

    walmart_mcp = MCPServer(
        'walmart-store-ops',
        instructions=(
            'Walmart India Store Operations MCP Server. '
            'Exposes real-time weather intelligence and market demand signals '
            'for store management decisions at WMT stores across India.'
        ),
    )

    @walmart_mcp.tool()
    def get_store_weather(city: str, country_code: str = 'IN') -> dict:
        """Get current weather at a Walmart India store location for demand planning."""
        return fetch_weather(city, country_code)

    @walmart_mcp.tool()
    def search_demand_trends(query: str, max_results: int = 3) -> dict:
        """Search live retail demand trends and market signals for store planning."""
        return fetch_demand_trends(query, max_results)

    @walmart_mcp.tool()
    def get_store_info(store_id: str) -> dict: # store_id="WMT-0511"
        """Retrieve operational metadata for a Walmart India store."""
        registry = {
            'WMT-2847': {'location': 'Bengaluru, Karnataka', 'format': 'Supercenter',
                         'departments': 32, 'daily_queries': 1200, 'region': 'South India'},
            'WMT-1023': {'location': 'Mumbai, Maharashtra',  'format': 'Supercenter',
                         'departments': 28, 'daily_queries': 1800, 'region': 'West India'},
            'WMT-0511': {'location': 'Delhi NCR',            'format': 'Supercenter',
                         'departments': 35, 'daily_queries': 2100, 'region': 'North India'},
            'WMT-3302': {'location': 'Hyderabad, Telangana', 'format': 'Neighborhood Market',
                         'departments': 18, 'daily_queries':  620, 'region': 'South India'},
        }
        return registry.get(store_id, {'error': f'Store {store_id} not found in registry'})

    def _annotation_to_json_type(annotation) -> str:
        # Convert Python type hints into simple JSON schema types.
        return {
            str: 'string',
            int: 'integer',
            float: 'number',
            bool: 'boolean',
            dict: 'object',
            list: 'array',
        }.get(annotation, 'string')

    def _build_mcp_tool_entry(func) -> dict:
        # Build a small machine-readable schema for each MCP tool.
        sig = inspect.signature(func) # def get_store_weather(city: str, country_code: str = 'IN')
        properties = {}
        required = []

        for param_name, param in sig.parameters.items():
            if param.kind not in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY):
                continue

            prop = {'type': _annotation_to_json_type(param.annotation)}
            if param.default is not inspect._empty:
                prop['default'] = param.default
            else:
                required.append(param_name)
            properties[param_name] = prop

        return {
            'name': func.__name__,
            'description': inspect.getdoc(func) or '',
            'inputSchema': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
            'handler': func,
        }

    # Keep the registered MCP tools in one place for easy lookup below.
    MCP_TOOL_REGISTRY = {
        tool['name']: tool
        for tool in [
            _build_mcp_tool_entry(get_store_weather),
            _build_mcp_tool_entry(search_demand_trends),
            _build_mcp_tool_entry(get_store_info),
        ]
    }

    def call_mcp_tool(name: str, args: dict) -> dict:
        # Run an MCP tool by name using the saved registry entry.
        return MCP_TOOL_REGISTRY[name]['handler'](**args)

    MCP_AVAILABLE = True
    print(f'FastMCP server created: {walmart_mcp.name}')
    print()
    print('Tools registered on the MCP server:')
    print('  - get_store_weather    (live data: OpenWeatherMap API)')
    print('  - search_demand_trends (live data: Tavily Search API)')
    print('  - get_store_info       (store registry -- note: 3 tools vs 2 for REST)')
    print()
    print('Key difference from REST: tool descriptions live here on the server,')
    print('not in the developer\'s agent code. Any MCP client discovers them automatically.')

except ImportError as e:
    print(f'mcp package not installed: {e}')
    print('Install: pip install mcp --break-system-packages')
    print()
    print('The REST section above is fully functional without this package.')
    print('FastMCP cells below will be skipped gracefully.')

In [ ]:
# Now combine MCP tool schemas, MCP tool execution, and the LLM in one loop.
# This mirrors the REST agent flow, but the tool schema comes from the server side.
# Step 3: Full MCP Agent -- schema from server registry, tool execution from MCP server definitions

if MCP_AVAILABLE:
    def walmart_mcp_agent(query: str) -> dict:
        # Run a full MCP-style agent loop using the server-owned schema.
        start = time.time()

        openai_tools = [
            {
                'type': 'function',
                'function': {
                    'name':        t['name'],
                    'description': t['description'],
                    'parameters':  t['inputSchema'],
                },
            }
            for t in MCP_TOOL_REGISTRY.values()
        ]

        messages = [
            {
                'role': 'system',
                'content': (
                    f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                    'Tools are provided by the Walmart MCP server. '
                    'Use all relevant tools to give a complete, data-driven recommendation.'
                ),
            },
            {'role': 'user', 'content': query},
        ]

        tools_called = []
        total_in = total_out = 0
        final_answer = ''

        # Keep going until the model stops asking for tool calls.
        for _ in range(6):
            resp = client.chat.completions.create(
                model='gpt-4o-mini', messages=messages, tools=openai_tools,
                tool_choice='auto', temperature=0, max_tokens=500,
            )
            total_in  += resp.usage.prompt_tokens
            total_out += resp.usage.completion_tokens
            msg = resp.choices[0].message

            if not msg.tool_calls:
                final_answer = msg.content.strip()
                break

            messages.append(msg)

            for tc in msg.tool_calls:
                args   = json.loads(tc.function.arguments)
                result = call_mcp_tool(tc.function.name, args)
                tools_called.append({'tool': tc.function.name, 'args': args})
                messages.append({
                    'role':         'tool',
                    'tool_call_id': tc.id,
                    'content':      json.dumps(result),
                })

        latency = time.time() - start
        cost    = (total_in * 0.15 + total_out * 0.60) / 1_000_000
        return {
            'answer':       final_answer,
            'tools_called': tools_called,
            'latency_sec':  round(latency, 2),
            'cost_usd':     round(cost, 6),
            'tokens_in':    total_in,
            'tokens_out':   total_out,
            'protocol':     'MCP',
        }

    mcp_result = walmart_mcp_agent(STORE_QUERY)

    print('Tools called via MCP protocol:')
    for t in mcp_result['tools_called']:
        print(f'  {t["tool"]}({json.dumps(t["args"])})')
    print()
    print('MCP Agent Answer (based on live data):')
    print('-' * 70)
    print(mcp_result['answer'])
    print('-' * 70)
    print(f'Latency : {mcp_result["latency_sec"]}s')
    print(f'Cost    : ${mcp_result["cost_usd"]}')
    print(f'Protocol: MCP (schema auto-discovered from FastMCP server)')

else:
    mcp_result = {**rest_result, 'protocol': 'MCP (fallback -- install mcp package)'}
    print('mcp package not installed. Showing REST result as reference.')
    print(f'Answer: {mcp_result["answer"][:300]}...')

In [ ]:
STORE_QUERY

## Section 3: Framework Selection -- LangChain vs LangGraph vs Python-only

Framework selection is a TCO (Total Cost of Ownership) decision, not a features decision.
Every framework adds capabilities and adds cost: dependency risk, debugging complexity, version lock, and team learning curve.

All three implementations below use the **same real live data** (OpenWeatherMap + Tavily).
The difference is in how each framework manages state, control flow, and observability.

In [ ]:
# This version uses plain Python only: no tool framework, no graph, just direct calls.
# It is the simplest option, but the developer must control every step manually.
def python_only_agent(query: str) -> dict:
    # Python-only: full control, minimum abstraction.
    # Developer manually fetches data before the LLM call -- no dynamic tool calling.
    start = time.time()

    weather = fetch_weather(STORE_CITY)
    demand  = fetch_demand_trends(f'retail product demand {STORE_CITY} India', max_results=2)

    context = (
        f'Live weather for {STORE_CITY}: {json.dumps(weather)}. '
        f'Market demand signals: {demand["answer"]}'
    )

    resp = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {
                'role': 'system',
                'content': (
                    f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
                    f'Current live data: {context}'
                ),
            },
            {'role': 'user', 'content': query},
        ],
        temperature=0, max_tokens=400,
    )

    latency = time.time() - start
    ti, to  = resp.usage.prompt_tokens, resp.usage.completion_tokens
    return {
        'answer':      resp.choices[0].message.content.strip(),
        'latency_sec': round(latency, 2),
        'cost_usd':    round((ti * 0.15 + to * 0.60) / 1_000_000, 6),
        'framework':   'Python-only',
        'data_fetch':  'Manual pre-fetch (LLM cannot request more data dynamically)',
        'loc_approx':  30,
    }

py_result = python_only_agent(STORE_QUERY)
print('Python-only Agent Answer:')
print('-' * 60)
print(py_result['answer'][:450])
print('-' * 60)
print(f'Latency: {py_result["latency_sec"]}s | Framework: Python-only')
print(f'Note   : {py_result["data_fetch"]}')

In [ ]:
STORE_QUERY

In [ ]:
# %pip install langchain langchain-openai -q

In [ ]:
# This version uses LangChain to manage tools and the agent loop for us.
# It reduces boilerplate, but adds another abstraction layer to learn and debug.
LC_AVAILABLE = False
lc_latency   = None

try:
    from langchain_openai import ChatOpenAI # LLM / Brain
    from langchain_core.prompts import ChatPromptTemplate # Prompt/Ears
    from langchain_core.tools import tool # Hands
    from langchain.agents import create_tool_calling_agent, AgentExecutor
    # create_tool_calling_agent - Combining LLM+Prompt+Tools
    # AgentExecutor - Running everything

    @tool
    def lc_get_weather(city: str) -> str:
        """Get real-time weather at a Walmart India store location for demand planning."""
        return json.dumps(fetch_weather(city))

    @tool
    def lc_search_trends(query: str) -> str:
        """Search for real-time retail demand trends and market signals using Tavily."""
        return json.dumps(fetch_demand_trends(query, max_results=2))

    lc_llm   = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    lc_tools = [lc_get_weather, lc_search_trends]

    lc_prompt = ChatPromptTemplate.from_messages([
        ('system',
         f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
         'Use tools to retrieve live data before making recommendations.'),
        ('human', '{input}'),
        ('placeholder', '{agent_scratchpad}'),
    ])

    lc_agent    = create_tool_calling_agent(lc_llm, lc_tools, lc_prompt)
    lc_executor = AgentExecutor(agent=lc_agent, tools=lc_tools, verbose=False, max_iterations=6)

    start       = time.time()
    lc_response = lc_executor.invoke({'input': STORE_QUERY})
    # lc_response = lc_executor.invoke({'input': "What is the current date and time in India?"})
    lc_latency  = time.time() - start

    print('LangChain Agent Answer (real tools via LCEL + AgentExecutor):')
    print('-' * 60)
    print(lc_response['output'][:450])
    print('-' * 60)
    print(f'Latency: {lc_latency:.2f}s | Framework: LangChain')
    LC_AVAILABLE = True

except ImportError as e:
    print(f'LangChain not installed: {e}')
    print('Install: pip install langchain langchain-openai --break-system-packages')

In [ ]:
# # This version uses LangChain to manage tools and the agent loop for us.
# # It reduces boilerplate, but adds another abstraction layer to learn and debug.
# LC_AVAILABLE = False
# lc_latency   = None
# import time

# try:
#     from langchain_openai import ChatOpenAI # Brain - LLM
#     from langchain_core.prompts import ChatPromptTemplate # Mouth - prompt template
#     from langchain_core.tools import tool # Hands - Tools
#     from langchain.agents import create_tool_calling_agent, AgentExecutor
#     # create_tool_calling_agent = Combine (LLM + Tools + PromptTemplate)

#     @tool
#     def lc_get_weather(city: str) -> str:
#         """Get real-time weather at a Walmart India store location for demand planning."""
#         return json.dumps(fetch_weather(city))

#     @tool
#     def lc_search_trends(query: str) -> str:
#         """Search for real-time retail demand trends and market signals using Tavily."""
#         return json.dumps(fetch_demand_trends(query, max_results=2))

#     @tool
#     def get_current_utc_date_and_time() -> str:
#         """Get the current UTC date and time."""
#         return json.dumps({'utc_datetime': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())})

#     lc_llm   = ChatOpenAI(model='gpt-4o', temperature=0)
#     lc_tools = [lc_get_weather, lc_search_trends, get_current_utc_date_and_time]

#     lc_prompt = ChatPromptTemplate.from_messages([
#         ('system',
#          f'You are an AI assistant for Walmart India store {STORE_ID} in {STORE_CITY}. '
#          'Use tools to retrieve live data before making recommendations.'),
#         ('human', '{input}'),
#         ('placeholder', '{agent_scratchpad}'),
#     ])

#     lc_agent    = create_tool_calling_agent(lc_llm, lc_tools, lc_prompt)
#     lc_executor = AgentExecutor(agent=lc_agent, tools=lc_tools, verbose=True, max_iterations=6)

#     start       = time.time()
#     # lc_response = lc_executor.invoke({'input': "What is the current date and time in India?"})
#     lc_response = lc_executor.invoke({'input': "What is the current date and time in India?"})
#     # lc_response = lc_executor.invoke({'input': "I am in the IST time zone currently training the Walmart Folkz and my training is going to get over at 11 PM. IST. Can you tell me how much more time do I have to teach??"})
#     lc_latency  = time.time() - start

#     print('LangChain Agent Answer (real tools via LCEL + AgentExecutor):')
#     print('-' * 60)
#     print(lc_response['output'][:450])
#     print('-' * 60)
#     print(f'Latency: {lc_latency:.2f}s | Framework: LangChain')
#     LC_AVAILABLE = True

# except ImportError as e:
#     print(f'LangChain not installed: {e}')
#     print('Install: pip install langchain langchain-openai --break-system-packages')

In [ ]:
# %pip install -U langgraph langchain-core langchain-openai


In [ ]:
# %pip install langgraph -q